<a href="https://colab.research.google.com/github/Mayk17/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
print('Menu Detector')


Menu Detector


In [13]:
#---------------------
# Import Libraries
#---------------------

from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np


In [14]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
#Define dataset Path

DATASET_PATH = '/content/drive/MyDrive/food_dataset'
print("dataset_path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog", # label grouping | class consolidation
    "uzbek_pilaf": "uzbek_pilaf",
    "kebab shashlik": "kebab shashlik"
}

CLASSES = ["hamburger", "hot_dog", "uzbek_pilaf", "kebab shashlik"]
CLASS_TO_IDX = {cls: i for i , cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset_path: /content/drive/MyDrive/food_dataset
4
{'hamburger': 0, 'hot_dog': 1, 'uzbek_pilaf': 2, 'kebab shashlik': 3}


In [27]:
# ------------
# Custom DataSet Class
# ------------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.image = images
    self.labels = labels
    self.transform = transform

    def __len__(self):
      print('images_lenght', len(self.images))
      return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('img_path', img_path)
        label = self.labels[idx]
        print('label', label)

        try:
          image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
          print(f"Skipping broken image: {img_path}")
          return self.__getitem__(idx + 1) % len(self.images)
        if self.transform:
          image = self.transform(image)
        return image, label






In [31]:
#------------
# Gather and Split Data
#-----------

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class) # /content/drive/MyDrive/food_dataset/hamburger
  print(class_path)
  if not os.path.exists(class_path):
     print(f"Warning: {class_path} not found")
     continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food_dataset/hamburger/adsd.jpg
        all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split] # 1000 / 800 train data val_data
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print('all_images', len(all_images))

# train_data 대신 올바르게 분리된 train_images를 넘겨줍니다.
dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[1]


/content/drive/MyDrive/food_dataset/hamburger
/content/drive/MyDrive/food_dataset/hot_dog
/content/drive/MyDrive/food_dataset/uzbek_pilaf
/content/drive/MyDrive/food_dataset/kebab shashlik
all_images 207
165
